# Extract metrics

In [3]:
"""
Aggregate UNet test prediction metrics into a single summary CSV.
Only reads from test_predictions folders.
Supports both single-site and multi-site models.

Multi-site aggregation uses MICRO-AVERAGING:
- Sums TP, FP, FN, TN across all reserves
- Calculates metrics from totals (proper way to treat as one dataset)
"""
import os
import re
import csv
from pathlib import Path
import pandas as pd

def parse_model_folder_name(folder_name):
    """Parse model folder name like 'MW10_LR0_02_BCEDICE' into components."""
    pattern = r'MW(\d+)_LR([0-9e_\-]+)_(.+)'
    match = re.match(pattern, folder_name)
    
    if match:
        weight = int(match.group(1))
        lr_str = match.group(2).replace('_', '.').replace('.-', 'e-')
        loss = match.group(3)
        try:
            lr = float(lr_str)
        except:
            lr = lr_str
        return {'maire_weight': weight, 'learning_rate': lr, 'loss_function': loss}
    return None


def parse_dataset_folder_name(folder_name):
    """Parse dataset folder name like 'MS_REL_ESK' or 'MS_REL_ESK_BUS_KAU_HAM' into components."""
    band_types = ['MS_REL_RENDVI', 'MS_ABS_RENDVI', 'MS_REL', 'MS_ABS', 'IND_REL', 'IND_ABS', 'RGB']
    
    for band_type in band_types:
        if folder_name.startswith(band_type + '_'):
            reserves_part = folder_name[len(band_type) + 1:]
            reserves = reserves_part.split('_')
            return {'band_combination': band_type, 'reserves': reserves}
    return None


def read_metrics_csv(csv_path):
    """Read metrics from a CSV file, including confusion matrix values if available."""
    try:
        with open(csv_path, 'r') as f:
            reader = csv.DictReader(f)
            row = next(reader)
            result = {
                'mIoU': float(row.get('mIoU', 0)),
                'Maire Precision': float(row.get('Maire Precision', 0)),
                'Maire Recall': float(row.get('Maire Recall', 0)),
                'Maire F1': float(row.get('Maire F1', 0)),
                'Maire IoU': float(row.get('Maire IoU', 0)),
            }
            # Read confusion matrix values if available (for micro-averaging)
            if 'TP' in row:
                result['TP'] = int(row['TP'])
                result['FP'] = int(row['FP'])
                result['FN'] = int(row['FN'])
                result['TN'] = int(row['TN'])
            return result
    except Exception as e:
        return None


def calculate_metrics_from_confusion(TP, FP, FN, TN):
    """Calculate metrics from confusion matrix values (micro-averaging)."""
    eps = 1e-8
    precision = TP / (TP + FP + eps)
    recall = TP / (TP + FN + eps)
    f1 = 2 * precision * recall / (precision + recall + eps)
    iou = TP / (TP + FP + FN + eps)
    bg_iou = TN / (TN + FP + FN + eps)
    miou = (bg_iou + iou) / 2
    return {
        'mIoU': miou,
        'Maire Precision': precision,
        'Maire Recall': recall,
        'Maire F1': f1,
        'Maire IoU': iou
    }


def aggregate_results(models_dir, model_type='best_maire_f1', zone_type='unet_test_zone', aggregate_multi_site=True):
    """
    Aggregate all test prediction metrics from test_predictions folders.
    
    Args:
        models_dir: Path to the models directory (e.g., 5_3_unet/models)
        model_type: 'best_maire_f1' or 'best_iou'
        zone_type: 'unet_test_zone' or 'bbox'
        aggregate_multi_site: If True, MICRO-AVERAGE across reserves (sum TP/FP/FN/TN, then calculate)
                              This properly treats multi-site as ONE dataset.
                              If False, show each reserve separately.
    """
    results = []
    models_path = Path(models_dir)
    
    # Iterate through dataset folders (e.g., MS_REL_ESK, RGB_BUS)
    for dataset_folder in sorted(models_path.iterdir()):
        if not dataset_folder.is_dir():
            continue
        
        dataset_info = parse_dataset_folder_name(dataset_folder.name)
        if not dataset_info:
            continue
        
        reserves = dataset_info['reserves']
        is_multi_site = len(reserves) > 1
        
        # Iterate through model folders (e.g., MW10_LR0_02_BCEDICE)
        for model_folder in sorted(dataset_folder.iterdir()):
            if not model_folder.is_dir():
                continue
            
            model_info = parse_model_folder_name(model_folder.name)
            if not model_info:
                continue
            
            # ONLY look in test_predictions folder
            test_preds_dir = model_folder / 'test_predictions'
            if not test_preds_dir.exists():
                continue
            
            if is_multi_site:
                # Multi-site: look for reserve-specific files (new format)
                # Format: best_maire_f1_ESK_unet_test_zone_metrics.csv
                reserve_metrics = []
                for reserve in reserves:
                    metrics_csv = test_preds_dir / f'{model_type}_{reserve}_{zone_type}_metrics.csv'
                    metrics = read_metrics_csv(metrics_csv)
                    if metrics:
                        reserve_metrics.append((reserve, metrics))
                        
                        # Add individual reserve row if not aggregating
                        if not aggregate_multi_site:
                            row = {
                                'Reserve(s)': reserve,
                                'Training Reserves': '_'.join(reserves),
                                'Band Combination': dataset_info['band_combination'],
                                'Maire Weight': model_info['maire_weight'],
                                'Learning Rate': model_info['learning_rate'],
                                'Loss Function': model_info['loss_function'],
                                'mIoU': metrics['mIoU'],
                                'Maire Precision': metrics['Maire Precision'],
                                'Maire Recall': metrics['Maire Recall'],
                                'Maire F1': metrics['Maire F1'],
                                'Maire IoU': metrics['Maire IoU'],
                            }
                            results.append(row)
                
                # MICRO-AVERAGE: Sum confusion matrices, then calculate metrics
                if aggregate_multi_site and reserve_metrics:
                    # Check if we have confusion matrix values
                    has_confusion = all('TP' in m for _, m in reserve_metrics)
                    
                    if has_confusion:
                        # Proper micro-averaging: sum TP/FP/FN/TN across all reserves
                        total_TP = sum(m['TP'] for _, m in reserve_metrics)
                        total_FP = sum(m['FP'] for _, m in reserve_metrics)
                        total_FN = sum(m['FN'] for _, m in reserve_metrics)
                        total_TN = sum(m['TN'] for _, m in reserve_metrics)
                        
                        micro_metrics = calculate_metrics_from_confusion(total_TP, total_FP, total_FN, total_TN)
                        
                        row = {
                            'Reserve(s)': '_'.join(reserves),
                            'Band Combination': dataset_info['band_combination'],
                            'Maire Weight': model_info['maire_weight'],
                            'Learning Rate': model_info['learning_rate'],
                            'Loss Function': model_info['loss_function'],
                            'mIoU': micro_metrics['mIoU'],
                            'Maire Precision': micro_metrics['Maire Precision'],
                            'Maire Recall': micro_metrics['Maire Recall'],
                            'Maire F1': micro_metrics['Maire F1'],
                            'Maire IoU': micro_metrics['Maire IoU'],
                            'Num Reserves': len(reserve_metrics),
                            'Aggregation': 'micro',
                        }
                    else:
                        # Fallback to macro-averaging if no confusion matrix available
                        avg_metrics = {
                            'mIoU': sum(m['mIoU'] for _, m in reserve_metrics) / len(reserve_metrics),
                            'Maire Precision': sum(m['Maire Precision'] for _, m in reserve_metrics) / len(reserve_metrics),
                            'Maire Recall': sum(m['Maire Recall'] for _, m in reserve_metrics) / len(reserve_metrics),
                            'Maire F1': sum(m['Maire F1'] for _, m in reserve_metrics) / len(reserve_metrics),
                            'Maire IoU': sum(m['Maire IoU'] for _, m in reserve_metrics) / len(reserve_metrics),
                        }
                        row = {
                            'Reserve(s)': '_'.join(reserves),
                            'Band Combination': dataset_info['band_combination'],
                            'Maire Weight': model_info['maire_weight'],
                            'Learning Rate': model_info['learning_rate'],
                            'Loss Function': model_info['loss_function'],
                            'mIoU': avg_metrics['mIoU'],
                            'Maire Precision': avg_metrics['Maire Precision'],
                            'Maire Recall': avg_metrics['Maire Recall'],
                            'Maire F1': avg_metrics['Maire F1'],
                            'Maire IoU': avg_metrics['Maire IoU'],
                            'Num Reserves': len(reserve_metrics),
                            'Aggregation': 'macro (no confusion matrix)',
                        }
                    results.append(row)
            else:
                # Single-site: try both old format and new format
                reserve = reserves[0]
                # New format: best_maire_f1_ESK_unet_test_zone_metrics.csv
                metrics_csv = test_preds_dir / f'{model_type}_{reserve}_{zone_type}_metrics.csv'
                metrics = read_metrics_csv(metrics_csv)
                
                # Fallback to old format: best_maire_f1_unet_test_zone_metrics.csv
                if not metrics:
                    metrics_csv = test_preds_dir / f'{model_type}_{zone_type}_metrics.csv'
                    metrics = read_metrics_csv(metrics_csv)
                
                if metrics:
                    row = {
                        'Reserve(s)': reserve,
                        'Band Combination': dataset_info['band_combination'],
                        'Maire Weight': model_info['maire_weight'],
                        'Learning Rate': model_info['learning_rate'],
                        'Loss Function': model_info['loss_function'],
                        'mIoU': metrics['mIoU'],
                        'Maire Precision': metrics['Maire Precision'],
                        'Maire Recall': metrics['Maire Recall'],
                        'Maire F1': metrics['Maire F1'],
                        'Maire IoU': metrics['Maire IoU'],
                    }
                    results.append(row)
    
    return pd.DataFrame(results)


# =============================================================================
# CONFIGURATION - Adjust these parameters
# =============================================================================
models_dir = r'../../5_3_unet/models'
model_type = 'best_maire_f1'  # Options: 'best_maire_f1', 'best_iou'
zone_type = 'unet_test_zone'  # Options: 'unet_test_zone', 'bbox'
aggregate_multi_site = True   # True = MICRO-AVERAGE (sum TP/FP/FN/TN), False = show each reserve

# Run aggregation
df = aggregate_results(models_dir, model_type=model_type, zone_type=zone_type, aggregate_multi_site=aggregate_multi_site)

# Save to CSV
output_csv = r'../../2_1_figures/results/summary.csv'
df.to_csv(output_csv, index=False)
print(f"Saved to {output_csv}")

# Display results
print(f"Found {len(df)} model results")
print(f"Model type: {model_type}, Zone: {zone_type}, Aggregate multi-site: {aggregate_multi_site}")
df.sort_values(by='Maire IoU', ascending=False, inplace=True)
df.head(50)

Saved to ../../2_1_figures/results/summary.csv
Found 135 model results
Model type: best_maire_f1, Zone: unet_test_zone, Aggregate multi-site: True


,Reserve(s),Band Combination,Maire Weight,Learning Rate,Loss Function,mIoU,Maire Precision,Maire Recall,Maire F1,Maire IoU,Num Reserves,Aggregation
19,BUS,MS_REL,1,0.02000,DICE,0.817059,0.744158,0.883882,0.808024,0.677886,NaN,NaN
18,BUS,MS_REL,1,0.02000,BCEDICE,0.809736,0.723442,0.893825,0.799659,0.666193,NaN,NaN
76,BUS,RGB,10,0.02000,BCE,0.788452,0.682430,0.873725,0.766320,0.621166,NaN,NaN
77,BUS,RGB,10,0.02000,BCEDICE,0.779004,0.741615,0.758011,0.749723,0.599646,NaN,NaN
82,BUS,RGB,1,0.02000,DICE,0.766848,0.638939,0.874289,0.738312,0.585178,NaN,NaN
81,BUS,RGB,1,0.02000,BCEDICE,0.755226,0.621730,0.861949,0.722393,0.565427,NaN,NaN
14,BUS,MS_REL,10,0.02000,BCEDICE,0.693506,0.511322,0.912091,0.655287,0.487306,NaN,NaN
38,ESK,MS_REL,50,0.02000,BCEDICE,0.723479,0.522823,0.869510,0.653004,0.484786,NaN,NaN
87,BUS,RGB,50,0.02000,BCEDICE,0.703079,0.523925,0.859557,0.651029,0.482611,NaN,NaN
73,ESK,MS_REL_RENDVI,10,0.02000,BCEDICE,0.717424,0.511790,0.865701,0.643281,0.474145,NaN,NaN
